In [1]:
import pandas as pd

In [2]:
years = [2020, 2021, 2022, 2023, 2024, 2025]
url_link = 'https://www.basketball-reference.com/leagues/NBA_{}_per_game.html'

points_threshold = 20  # Points threshold for All-Star selection
team_high_scorers_outliers = {}

all_outliers = []
all_normal = []

for year in years:
    url = url_link.format(year)
    df = pd.read_html(url, header=0)[0]
    df = df.drop(df[df['Age'] == 'Age'].index)

    df['Year'] = year
    df['PTS'] = pd.to_numeric(df['PTS'], errors='coerce')

    # Add AllStarSelection column
    df['AllStarSelection'] = (df['PTS'] >= points_threshold).astype(int)

    avg_pts_per_game = df['PTS'].mean()
    print(f"Average points per game in {year}: {avg_pts_per_game:.2f}")

    Q1 = df['PTS'].quantile(0.25)
    Q3 = df['PTS'].quantile(0.75)
    IQR = Q3 - Q1
    high_threshold = Q3 + 1.5 * IQR

    high_outliers = df[df['PTS'] > high_threshold]
    normal_data = df[df['PTS'] <= high_threshold]

    all_outliers.append(high_outliers)
    all_normal.append(normal_data)

    for index, row in high_outliers.iterrows():
        player_name = row['Player']
        team = row['Team']
        if team in team_high_scorers_outliers:
            if player_name in team_high_scorers_outliers[team]:
                team_high_scorers_outliers[team][player_name].add(year)
            else:
                team_high_scorers_outliers[team][player_name] = {year}
        else:
            team_high_scorers_outliers[team] = {player_name: {year}}

# Combine all years' data
outliers_df = pd.concat(all_outliers, ignore_index=True)
normal_df = pd.concat(all_normal, ignore_index=True)

# Save to CSV
outliers_df.to_csv("Raw/nba_high_scorers_outliers.csv", index=False)
normal_df.to_csv("Raw/nba_normal_scorers.csv", index=False)

# Optional summary print
print("\nHigh Scorers Outliers Grouped by Team:")
for team, player_outliers in team_high_scorers_outliers.items():
    print(f"\nTeam: {team}")
    for player, outlier_years in player_outliers.items():
        outlier_years_str = ', '.join(map(str, outlier_years))
        print(f"  Player: {player}, Outlier Years: {outlier_years_str}")

Average points per game in 2020: 8.46
Average points per game in 2021: 8.62
Average points per game in 2022: 7.81
Average points per game in 2023: 8.86
Average points per game in 2024: 8.02
Average points per game in 2025: 8.56

High Scorers Outliers Grouped by Team:

Team: HOU
  Player: James Harden, Outlier Years: 2020, 2021
  Player: Russell Westbrook, Outlier Years: 2020

Team: WAS
  Player: Bradley Beal, Outlier Years: 2020, 2021, 2022, 2023
  Player: Kristaps Porziņģis, Outlier Years: 2022, 2023

Team: POR
  Player: Damian Lillard, Outlier Years: 2020, 2021, 2022, 2023
  Player: Anfernee Simons, Outlier Years: 2024

Team: ATL
  Player: Trae Young, Outlier Years: 2020, 2021, 2022, 2023, 2024, 2025

Team: MIL
  Player: Giannis Antetokounmpo, Outlier Years: 2020, 2021, 2022, 2023, 2024, 2025
  Player: Damian Lillard, Outlier Years: 2024, 2025

Team: DAL
  Player: Luka Dončić, Outlier Years: 2020, 2021, 2022, 2023, 2024, 2025
  Player: Kyrie Irving, Outlier Years: 2024, 2025, 2023

T

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 736 entries, 0 to 735
Data columns (total 33 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Rk                735 non-null    float64
 1   Player            736 non-null    object 
 2   Age               735 non-null    float64
 3   Team              735 non-null    object 
 4   Pos               735 non-null    object 
 5   G                 735 non-null    float64
 6   GS                735 non-null    float64
 7   MP                735 non-null    float64
 8   FG                735 non-null    float64
 9   FGA               735 non-null    float64
 10  FG%               732 non-null    float64
 11  3P                735 non-null    float64
 12  3PA               735 non-null    float64
 13  3P%               691 non-null    float64
 14  2P                735 non-null    float64
 15  2PA               735 non-null    float64
 16  2P%               725 non-null    float64
 1

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def scrape_hoopshype_salaries(start_year, end_year):
    all_data = []

    headers = {
        'User-Agent': 'Mozilla/5.0'
    }

    for year in range(start_year, end_year + 1):
        next_year = year + 1
        url = f'https://hoopshype.com/salaries/players/{year}-{next_year}/'
        print(f"Scraping {url}")
        response = requests.get(url, headers=headers)

        if response.status_code != 200:
            print(f"Failed to retrieve data for {year}-{next_year}")
            continue

        soup = BeautifulSoup(response.text, 'html.parser')
        table = soup.find('table', class_='hh-salaries-ranking-table')

        if not table:
            print(f"No table found for {year}-{next_year}")
            continue

        tbody = table.find('tbody')
        rows = tbody.find_all('tr')

        for row in rows:
            cols = row.find_all('td')
            if len(cols) >= 4:
                player = cols[1].get_text(strip=True)
                team = cols[2].get_text(strip=True)
                salary = cols[3].get_text(strip=True).replace('$', '').replace(',', '')
                try:
                    salary = float(salary)
                except ValueError:
                    salary = None

                all_data.append({
                    'Year': f"{year}-{next_year}",
                    'Player': player,
                    'Team': team,
                    'Salary': salary
                })

        time.sleep(1)  # Be respectful with a delay between requests

    df = pd.DataFrame(all_data)
    return df

# Example usage:
start_year = 2019
end_year = 2024
salaries_df = scrape_hoopshype_salaries(start_year, end_year)
salaries_df['Year'] = salaries_df['Year'].str.split('-').str[1].astype(int)

# Save to CSV
salaries_df.to_csv('Raw/nba_player_salaries_by_year.csv', index=False)


Scraping https://hoopshype.com/salaries/players/2019-2020/
Scraping https://hoopshype.com/salaries/players/2020-2021/
Scraping https://hoopshype.com/salaries/players/2021-2022/
Scraping https://hoopshype.com/salaries/players/2022-2023/
Scraping https://hoopshype.com/salaries/players/2023-2024/
Scraping https://hoopshype.com/salaries/players/2024-2025/
